# Held-out DAS-versus-network comparison result

This notebook is the first result checkpoint after the release-gated, time-only comparison. It is intentionally descriptive: it shows what matched in time, what remained DAS-only or network-only, and what is still scientifically pending. It does not assign earthquake identities, repeater families, false-discovery rates, slip rates, or stress drops. The 21 DAS-only rows require independent adjudication.

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'config' / 'heldout_das_network_comparison.json').is_file())
OUT = ROOT / 'outputs' / 'heldout_v2' / 'comparison'
REG = ROOT / 'outputs' / 'heldout_v2' / 'registration'

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

status = json.loads((OUT / 'time_only_comparison_status.json').read_text())
assert status['status'] == 'PASS'
assert status['time_only_output_row_count'] == 54
assert status['DAS_v2_candidate_count'] == 22
assert status['network_raw_candidate_count'] == 33
assert status['DAS_network_matched_pair_count'] == 1
assert status['DAS_only_candidate_count'] == 21
assert status['family_assignments_made'] == 0
assert status['time_only_comparison_sha256'] == sha256(OUT / 'das_network_time_only.csv')
assert status['network_context_sha256'] == sha256(OUT / 'das_network_network_context.csv')
print('PASS: released result hashes and scientific STOP gates verify')

In [ ]:
time_only = pd.read_csv(OUT / 'das_network_time_only.csv')
context = pd.read_csv(OUT / 'das_network_network_context.csv')
intervals = pd.read_csv(OUT / 'interval_summary.csv')
assert len(time_only) == len(context) == 54
assert len(intervals) == 12
assert set(time_only['comparison_membership']) == {'DAS_only', 'network_only', 'DAS+network'}
assert int((time_only['comparison_membership'] == 'DAS+network').sum()) == 1
assert int((time_only['comparison_membership'] == 'DAS_only').sum()) == 21
assert int((time_only['comparison_membership'] == 'network_only').sum()) == 32
assert time_only['absolute_time_difference_s'].dropna().max() <= 8.0
display(intervals[['interval_id', 'DAS_v2_candidate_count', 'network_raw_candidate_count', 'matched_pair_count', 'DAS_only_candidate_count', 'network_only_candidate_count']])

In [ ]:
matched = context[context['comparison_membership'] == 'DAS+network'].copy()
das_only = context[context['comparison_membership'] == 'DAS_only'].copy()
network_only = context[context['comparison_membership'] == 'network_only'].copy()
display(matched[['interval_id', 'comparison_time', 'DAS_candidate_id', 'network_union_candidate_id', 'absolute_time_difference_s']])
print(f'DAS-only rows pending independent adjudication: {len(das_only)}')
print(f'Network-only rows retained for context: {len(network_only)}')
assert das_only['DAS_independent_adjudication_status'].eq('pending_independent_catalog_forced_network_score_waveform_and_DAS_artifact_review').all()
assert context['repeater_family_assignment'].eq('not_assigned').all()

## Interpretation and next checkpoint

Only one DAS candidate is within 8 seconds of a network candidate, in heldout_08, with a 0.1004 s absolute difference. Twenty-one DAS candidates are not time-matched, but this is not yet a detection-extension result: independent adjudication must force the frozen network score at each DAS-only time, test catalog/regional-arrival associations, inspect DAS persistence/morphology and waveform evidence, and quantify interval-level uncertainty.

The next scientific checkpoint is an adjudication registration, not a threshold or matching-window sweep. It should predeclare which DAS-only rows receive waveform review, which network scores and catalog queries are allowed, how artifacts are rejected, and how family assignment is withheld unless independent evidence supports it.